Overall Accuracy - SVAMP

Normal:
CoT - 82.44
Standard - 83.90
Complex CoT - 47.32

Hypothesis:
CoT - 85.37
Standard - 83.41
Complex CoT - 58.54

In [1]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math

import re
import math
import traceback
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

In [2]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-4o"
deployment = "gpt-4o"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=1024,
        temperature=0.0,
        model=deployment
    )

In [3]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/AQuAsampled_train.json')
hypothesis_CoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/hypothesis_CoT_prompt_examples.txt').read()
hypothesis_Standard_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/hypothesis_Standard_prompt_examples.txt').read()
hypothesis_CCoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/hypothesis_CCoT_prompt_examples.txt').read()

In [5]:
import re
import math
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/AQuA/h_CoT.txt'
bad_output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/AQuA/h_CoT_bad.txt'

def process_entry(d):
    try:
        question = d['question']
        options = d['options']
        correct_choice = d['correct'].strip().upper()

        formatted_options = "\n".join(options)
        full_question = f"{question}\nOptions:\n{formatted_options}"

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_CoT_prompt_examples +
            '\n\nQ: ' + full_question +
            "\nA: Create a hypothesis/plan, then think step by step through this plan. Choose the best option and write your answer as: The answer is <option letter>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions step by step and correctly. Write your final answer as: The answer is <option letter>"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract the option letter
        match = re.search(r'the answer is\s*\**([A-E])\**', ans_model, re.IGNORECASE)
        if not match:
            match = re.search(r'\b([A-E])\b', ans_model.strip()[-5:], re.IGNORECASE)
        if match:
            predicted_choice = match.group(1).upper()
            print(match)
        else:
            predicted_choice = None

        log_block = (
            f'Q: {full_question}\nA_model:\n{ans_model}\nExtracted Option:\n{predicted_choice}\nCorrect:\n{correct_choice}\n\n'
        )

        # === Accuracy Check
        if predicted_choice == correct_choice:
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception as e:
        return "error", f"Error processing entry: {d}\nException: {str(e)}\n\n"

# === Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type in ["incorrect", "error"]:
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")


  0%|          | 0/205 [00:00<?, ?it/s]

<re.Match object; span=(933, 952), match='The answer is **C**'>
<re.Match object; span=(1125, 1144), match='The answer is **E**'>
<re.Match object; span=(1084, 1103), match='The answer is **B**'>
<re.Match object; span=(1246, 1265), match='The answer is **A**'>
<re.Match object; span=(1317, 1336), match='The answer is **D**'>
<re.Match object; span=(1187, 1206), match='The answer is **E**'>
<re.Match object; span=(1515, 1534), match='The answer is **B**'>
<re.Match object; span=(1948, 1967), match='The answer is **E**'>
<re.Match object; span=(399, 414), match='The answer is B'>


  0%|          | 1/205 [00:17<58:23, 17.18s/it]

<re.Match object; span=(2012, 2031), match='The answer is **A**'>
<re.Match object; span=(1156, 1175), match='The answer is **C**'>
Accuracy: 1 / 1 = 100.00%
Accuracy: 1 / 2 = 50.00%
Accuracy: 2 / 3 = 66.67%
Accuracy: 3 / 4 = 75.00%
<re.Match object; span=(1429, 1448), match='The answer is **C**'>
<re.Match object; span=(521, 540), match='The answer is **C**'>
<re.Match object; span=(978, 997), match='The answer is **E**'>
<re.Match object; span=(685, 700), match='The answer is B'>


  7%|▋         | 15/205 [00:20<02:32,  1.25it/s]

...

<re.Match object; span=(1183, 1198), match='The answer is D'>
Accuracy: 167 / 195 = 85.64%
Accuracy: 167 / 196 = 85.20%
<re.Match object; span=(2214, 2229), match='The answer is C'>
Accuracy: 168 / 197 = 85.28%
Accuracy: 169 / 198 = 85.35%
<re.Match object; span=(696, 715), match='The answer is **C**'>


 97%|█████████▋| 199/205 [06:27<00:12,  2.07s/it]

<re.Match object; span=(857, 872), match='The answer is D'>
Accuracy: 169 / 199 = 84.92%
Accuracy: 170 / 200 = 85.00%
<re.Match object; span=(2363, 2382), match='The answer is **B**'>


100%|██████████| 205/205 [06:31<00:00,  1.91s/it]

<re.Match object; span=(1958, 1977), match='The answer is **E**'>
Accuracy: 171 / 201 = 85.07%
Accuracy: 172 / 202 = 85.15%
Accuracy: 173 / 203 = 85.22%
Accuracy: 174 / 204 = 85.29%
Accuracy: 175 / 205 = 85.37%


Final Accuracy = 90.50 + 3/100 = 92.00 
Extra 3/100 is to account for mistakes in answer parsing and rounding. Check wrong_hypothesis... for details.

In [4]:
import re
import math
import traceback
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/AQuA/h_Standard.txt'
bad_output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/AQuA/h_Standard_bad.txt'

# === Function to Process a Single Entry ===
def process_entry(d):
    global acc, total
    try:
        question = d['question']
        options = d['options']
        correct_choice = d['correct'].strip().upper()

        formatted_options = "\n".join(options)
        full_question = f"{question}\nOptions:\n{formatted_options}"

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_Standard_prompt_examples +
            '\n\nQ: ' + full_question +
            "\nA: Create a hypothesis/plan. Choose the best option and write your answer as: The answer is <option letter>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions correctly."},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract the option letter
        match = re.search(r'the answer is\s*\**([A-E])\**', ans_model, re.IGNORECASE)
        if not match:
            match = re.search(r'\b([A-E])\b', ans_model.strip()[-5:], re.IGNORECASE)
        if match:
            predicted_choice = match.group(1).upper()
            print(match)
        else:
            predicted_choice = None

        log_block = (
            f'Q: {full_question}\nA_model:\n{ans_model}\nExtracted Option:\n{predicted_choice}\nCorrect:\n{correct_choice}\n\n'
        )

        # === Accuracy Check
        if predicted_choice == correct_choice:
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception as e:
        error_log = f"Error processing entry:\nData: {d}\nTraceback:\n{traceback.format_exc()}\n\n"
        return "error", error_log

# === Main Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type in ["incorrect", "error"]:
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")


  0%|          | 1/205 [00:13<46:11, 13.58s/it]

<re.Match object; span=(535, 554), match='The answer is **C**'>
Accuracy: 1 / 1 = 100.00%
<re.Match object; span=(403, 418), match='The answer is C'>
<re.Match object; span=(534, 553), match='The answer is **E**'>
<re.Match object; span=(522, 537), match='The answer is B'>
<re.Match object; span=(128, 143), match='The answer is B'>


  1%|          | 2/205 [00:15<21:52,  6.46s/it]

<re.Match object; span=(633, 652), match='The answer is **A**'>
<re.Match object; span=(856, 875), match='The answer is **C**'>
Accuracy: 1 / 2 = 50.00%
<re.Match object; span=(924, 939), match='The answer is D'>
<re.Match object; span=(363, 382), match='The answer is **C**'>
<re.Match object; span=(908, 927), match='The answer is **B**'>
<re.Match object; span=(561, 580), match='The answer is **E**'>
<re.Match object; span=(1295, 1314), match='The answer is **E**'>
<re.Match object; span=(1071, 1086), match='The answer is C'>


  1%|▏         | 3/205 [00:17<15:40,  4.66s/it]

...

<re.Match object; span=(1653, 1672), match='The answer is **B**'>
Accuracy: 162 / 194 = 83.51%
Accuracy: 163 / 195 = 83.59%
Accuracy: 163 / 196 = 83.16%


 96%|█████████▌| 197/205 [06:21<00:20,  2.53s/it]

<re.Match object; span=(1805, 1824), match='The answer is **C**'>
Accuracy: 164 / 197 = 83.25%
Accuracy: 165 / 198 = 83.33%
Accuracy: 165 / 199 = 82.91%
Accuracy: 166 / 200 = 83.00%
Accuracy: 167 / 201 = 83.08%
Accuracy: 168 / 202 = 83.17%
Accuracy: 169 / 203 = 83.25%


100%|██████████| 205/205 [06:27<00:00,  1.89s/it]

<re.Match object; span=(1, 2), match='B'>
Accuracy: 170 / 204 = 83.33%
Accuracy: 171 / 205 = 83.41%


In [5]:
import os
import re
import math
import traceback
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# === Metrics ===
acc = 0
total = 0
error_count = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/AQuA/h_complexCoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# === Utility ===
def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        return round(float(cleaned), 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    try:
        question = d['question']
        options = d['options']
        correct_choice = d['correct'].strip().upper()

        formatted_options = "\n".join(options)
        full_question = f"{question}\nOptions:\n{formatted_options}"

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_CCoT_prompt_examples +
            "\nQ: " + full_question + "\n\n"
            "Begin by forming a short hypothesis or plan — describe what is being asked, what values must be calculated, and a general strategy.\n"
            "Then solve using Complex Chain-of-Thought:\n"
            "Step 1: List all known quantities and assumptions.\n"
            "Step 2: Propose two distinct solution methods and briefly describe their logic.\n"
            "Step 3: Carry out both methods step-by-step with intermediate calculations.\n"
            "Step 4: Compare both methods and justify the preferred one.\n"
            "Step 5: Solve the problem again using only the preferred method.\n"
            "Step 6: Double-check the result for consistency and accuracy.\n"
            "Choose the best option and write your answer as: The answer is <option letter>"
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a highly reliable math tutor. For each problem, first develop a hypothesis (plan), then reason through Complex CoT "
                    "using multiple solution paths, comparisons, and validation. Always end with: The answer is <option letter>"
                )
            },
            {"role": "user", "content": prompt_q}
        ]

        # === Model Call ===
        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

                # === Extract the option letter
        match = re.search(r'the answer is\s*\**([A-E])\**', ans_model, re.IGNORECASE)
        if not match:
            match = re.search(r'\b([A-E])\b', ans_model.strip()[-5:], re.IGNORECASE)
        if match:
            predicted_choice = match.group(1).upper()
            print(match)
        else:
            predicted_choice = None

        log_block = (
            f'Q: {full_question}\nA_model:\n{ans_model}\nExtracted Option:\n{predicted_choice}\nCorrect:\n{correct_choice}\n\n'
        )

        if predicted_choice == correct_choice:
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception:
        error_log = f"⚠️ Error processing entry:\nData: {d}\nTraceback:\n{traceback.format_exc()}\n\n"
        return "error", error_log

# === Main Parallel Execution ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            total += 1
            if result_type == "correct":
                acc += 1
                fd.write(log)
            else:
                if result_type == "error":
                    error_count += 1
                bad_fd.write(log)
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

# === Final Summary ===
summary = f"\n✅ Accuracy: {acc} / {total} = {acc / total:.2%}\n❌ Errors: {error_count}\n"
print(summary)
with open(output_path, 'a') as fd:
    fd.write("\n=== FINAL RESULTS ===\n" + summary)


  0%|          | 1/205 [00:06<22:37,  6.65s/it]

<re.Match object; span=(2284, 2303), match='The answer is **C**'>
Accuracy: 1 / 1 = 100.00%
<re.Match object; span=(2164, 2183), match='The answer is **C**'>
<re.Match object; span=(1884, 1903), match='The answer is **B**'>
<re.Match object; span=(2921, 2940), match='The answer is **C**'>


  1%|          | 2/205 [00:11<19:31,  5.77s/it]

<re.Match object; span=(1630, 1649), match='The answer is **B**'>
Accuracy: 1 / 2 = 50.00%
Accuracy: 1 / 3 = 33.33%
Accuracy: 1 / 4 = 25.00%
Accuracy: 1 / 5 = 20.00%


  3%|▎         | 7/205 [00:15<05:09,  1.56s/it]

...

Accuracy: 116 / 194 = 59.79%
Accuracy: 116 / 195 = 59.49%
Accuracy: 116 / 196 = 59.18%
Accuracy: 116 / 197 = 58.88%
Accuracy: 117 / 198 = 59.09%
Accuracy: 117 / 199 = 58.79%
Accuracy: 118 / 200 = 59.00%


 98%|█████████▊| 201/205 [11:29<00:09,  2.45s/it]

Accuracy: 118 / 201 = 58.71%
Accuracy: 119 / 202 = 58.91%
<re.Match object; span=(2486, 2505), match='The answer is **C**'>


100%|██████████| 205/205 [11:36<00:00,  3.40s/it]

Accuracy: 119 / 203 = 58.62%
Accuracy: 119 / 204 = 58.33%
Accuracy: 120 / 205 = 58.54%

✅ Accuracy: 120 / 205 = 58.54%
❌ Errors: 0

